In [1]:
%pip install duckdb datashader colorcet 

Note: you may need to restart the kernel to use updated packages.


In [8]:
import duckdb
import datashader as ds
import datashader.transfer_functions as tf
import colorcet as cc
from PIL import Image
import os

Image.MAX_IMAGE_PIXELS = None

In [9]:
YEARS = [2012, 2013]
SPECIES = ["*"]
OUTPUT_DIR = "/mnt/shared_data/finflow/images/"
W, H = 6000, 3000

In [10]:
cvs = ds.Canvas(plot_width=W, plot_height=H, x_range=(-180, 180), y_range=(-90, 90))

for year in YEARS:
    df_gfw = duckdb.query(f"SELECT lon, lat, hours FROM read_parquet('/mnt/shared_data/finflow/gfw_raw/{year}/*.parquet')").to_df()
    agg_gfw = cvs.points(df_gfw, 'lon', 'lat', ds.sum('hours'))
    img_gfw = tf.shade(agg_gfw, cmap=cc.fire, how='log').to_pil().convert("RGBA")

    for spec in SPECIES:
        query_obis = f"""SELECT decimalLongitude as lon, decimalLatitude as lat FROM read_parquet('/mnt/shared_data/finflow/obis_raw/{spec}/*/*.parquet') 
                         WHERE year(eventDate) = {year} AND lon IS NOT NULL AND lat IS NOT NULL"""
        df_obis = duckdb.query(query_obis).to_df()
        
        combined = Image.open("/mnt/shared_data/finflow/images/base_map.png").resize((W, H)).convert("RGBA")
        #combined = Image.new("RGBA", (W, H), (0, 0, 0, 255))
        combined.alpha_composite(img_gfw)
        
        if not df_obis.empty:
            agg_obis = cvs.points(df_obis, 'lon', 'lat', ds.count())
            img_obis = tf.shade(agg_obis, cmap=["#90ee90", "#00ff00"], how='log').to_pil().convert("RGBA")
            combined.alpha_composite(img_obis)
        
        file_spec = spec.replace("*", "all").replace(" ", "_")
        save_path = os.path.join(OUTPUT_DIR, f"{year}_{file_spec}.png")
        combined.save(save_path)
        print(f"Saved: {year} - {spec}")

Saved: 2012 - *
Saved: 2013 - *
